# Warp resampling test-case inspector

Visual debugging aid for the warp golden-regression tests in `test/test_04_raster.py`
(`test_warp_resampling_golden`). When a case fails, run this notebook to *see* what changed: it
loads the same committed input raster the tests use (from `test.test_case_creator`), warps it with
the selected algorithm, and shows the source, the produced output, the committed golden, and their
difference side by side.

There are two test cases: an **integer** raster (`CASE = "int"`, all whole numbers) and a **float**
raster (`CASE = "float"`, every feature carries a fraction so the float code path is exercised).
Both always carry a nodata region, which is drawn blank (white) in the plots.

Input rasters live in `geokit/data/raster_data/input_data/resampling_input_<case>.tif` and golden
references in `geokit/data/raster_data/golden_regression_results/warp_resampling_<case>_<alg>.tif`.
The inputs are written by `python -m test.test_case_creator.create_rasters`; the golden references
are generated the first time the tests run. To regenerate after a deliberate GDAL/PROJ upgrade, run
the tests with `GEOKIT_REGEN_GOLDEN=1` and commit the updated files.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

# Make the repo importable: walk up from the notebook directory until we find the geokit package.
REPO_ROOT = os.getcwd()
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, "geokit")):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

from geokit import raster
from test.helpers import assert_raster_equal
from test.test_case_creator import load_test_raster, golden_raster_path, TEST_CASE_NAMES

print("repo root:", REPO_ROOT, "| cases:", TEST_CASE_NAMES)

## Pick a case

Set `CASE` to `"int"` or `"float"`, and `RESAMPLE_ALG` to the algorithm to inspect. Options:
`near`, `bilinear`, `cubic`, `cubicspline`, `lanczos`, `average`, `rms`, `mode`, `max`, `min`,
`med`, `q1`, `q3`, `sum`.

In [ ]:
CASE = "float"  # "int" or "float"
RESAMPLE_ALG = "cubic"
OUT_PIXEL = 200  # output resolution -> 2:1 downsample to 16x16

# Load the committed input raster (the exact file the tests run against) and read its nodata value.
SOURCE = load_test_raster(CASE)
NODATA = raster.rasterInfo(SOURCE).noData
print(f"case={CASE!r}  nodata={NODATA}  dtype={raster.rasterInfo(SOURCE).dtype}")

In [ ]:
def golden_path_for(alg):
    return golden_raster_path(CASE, alg)


def mask_nodata(mat):
    """Replace nodata with NaN so it renders blank, leaving valid data untouched."""
    return np.where(mat == NODATA, np.nan, mat)


def warp_out(alg):
    return raster.warp(SOURCE, resampleAlg=alg, pixelWidth=OUT_PIXEL, pixelHeight=OUT_PIXEL)


def load_case(alg):
    """Return (source_matrix, produced_matrix, golden_matrix_or_None, golden_path)."""
    out_mat = raster.extractMatrix(warp_out(alg))
    golden_path = golden_path_for(alg)
    gold_mat = raster.extractMatrix(golden_path) if os.path.isfile(golden_path) else None
    return raster.extractMatrix(SOURCE), out_mat, gold_mat, golden_path


src_mat, out_mat, gold_mat, golden_path = load_case(RESAMPLE_ALG)
print("source", src_mat.shape, "-> output", out_mat.shape)
print("golden:", golden_path, "(found)" if gold_mat is not None else "(MISSING - run pytest to generate)")

## Visualise source / output / golden / difference

Nodata pixels are blank (white).

In [ ]:
def show_case(alg):
    src_mat, out_mat, gold_mat, golden_path = load_case(alg)
    has_gold = gold_mat is not None
    ncols = 4 if has_gold else 2
    fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 4))

    def panel(ax, mat, title, **kw):
        h = ax.imshow(mat, interpolation="nearest", **kw)
        ax.set_title(title)
        fig.colorbar(h, ax=ax, fraction=0.046, pad=0.04)

    panel(axes[0], mask_nodata(src_mat), f"source {src_mat.shape}")
    panel(axes[1], mask_nodata(out_mat), f"warp '{alg}' {out_mat.shape}")
    if has_gold:
        panel(axes[2], mask_nodata(gold_mat), f"golden {gold_mat.shape}")
        diff = mask_nodata(out_mat).astype(float) - mask_nodata(gold_mat).astype(float)
        vmax = max(np.nanmax(np.abs(diff)), 1e-9)
        panel(
            axes[3],
            diff,
            f"produced - golden (max|d|={np.nanmax(np.abs(diff)):.3g})",
            cmap="RdBu",
            vmin=-vmax,
            vmax=vmax,
        )

    fig.suptitle(f"case={CASE!r}  resampleAlg='{alg}'")
    plt.tight_layout()
    plt.show()

    if has_gold:
        try:
            assert_raster_equal(golden_path, warp_out(alg))
            print(f"OK: '{CASE}/{alg}' matches the committed golden.")
        except AssertionError as exc:
            print(f"MISMATCH for '{CASE}/{alg}': {exc}")


show_case(RESAMPLE_ALG)

## Compare all algorithms at a glance

Uses the `CASE` selected above (nodata blank).

In [ ]:
ALGS = [
    "near",
    "bilinear",
    "cubic",
    "cubicspline",
    "lanczos",
    "average",
    "rms",
    "mode",
    "max",
    "min",
    "med",
    "q1",
    "q3",
    "sum",
]

fig, axes = plt.subplots(2, 7, figsize=(20, 6))
for ax, alg in zip(axes.ravel(), ALGS):
    out_mat = raster.extractMatrix(warp_out(alg))
    ax.imshow(mask_nodata(out_mat), interpolation="nearest")
    ax.set_title(alg)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"warp output per algorithm (case={CASE!r}, nodata blank)")
plt.tight_layout()
plt.show()